# 00. API 데이터 수집 노트북

이 노트북은 프로젝트의 **원본 데이터 수집 단계**만 다룬다.

실제 API 호출 코드는 `src/` 폴더에 그대로 유지하고, 이 노트북에서는 각 수집 스크립트를 순서대로 실행한다. 이렇게 하면 발표/검토용으로는 흐름을 쉽게 보여주고, 코드 유지보수는 `.py` 파일 한 곳에서 할 수 있다.

수집 대상:

- KOSIS 시군구 학령인구
- KOSIS 출생아수 및 인구이동
- KOSIS 합계출산율
- 소상공인시장진흥공단 상권 데이터
- 교육재정알리미/학교 관련 데이터

> API 키는 `.env`에 저장되어 있어야 하며, 노트북에서는 키 값을 출력하지 않는다.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

print("project root:", ROOT)
print("python:", sys.executable)

def run_script(script: str, *args: str) -> None:
    env = os.environ.copy()
    env.setdefault("PYTHONIOENCODING", "utf-8")
    env.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))
    path = ROOT / script
    if not path.exists():
        raise FileNotFoundError(path)
    print(f"\n=== RUN {script} ===")
    subprocess.run([sys.executable, str(path), *args], cwd=ROOT, check=True, env=env)

## 1. 환경 파일 확인

`.env` 파일이 있는지만 확인한다. API 키 값은 출력하지 않는다.

In [ ]:
env_path = ROOT / ".env"
print(".env exists:", env_path.exists())
print("config loader exists:", (ROOT / "src" / "config_loader.py").exists())

## 2. KOSIS 학령인구 수집

시군구 단위 연령대별 인구 및 학령인구 피처를 만든다. 회귀모델의 핵심 입력이다.

In [ ]:
# 시간이 걸릴 수 있음
# run_script("src/api/collect_national_kosis_population.py")

## 3. KOSIS 출생아수 및 인구이동 수집

출생 코호트 시나리오와 변화량 모델에서 중요한 입력이다.

In [ ]:
# run_script("src/api/collect_national_kosis_birth_migration.py")

## 4. KOSIS 합계출산율 수집

출산율은 장기 원인 변수다. 단기 회귀에서는 현재 학령인구가 더 직접적이지만, 발표에서는 출산율 경로 분석과 연결한다.

In [ ]:
# run_script("src/api/collect_national_kosis_fertility.py")

## 5. 전국 상권 데이터 수집 및 요약

학교 주변 생활 기반과 상권 취약도 피처를 만든다. 최종 위험등급에서 프로젝트 차별점으로 사용한다.

In [ ]:
# 수집량이 크므로 필요할 때만 실행
# run_script("src/api/collect_national_small_shop.py")

## 6. 교육재정알리미/학교 데이터 수집

현재 학교 목록, 학생수, 좌표, 폐교 관련 보조 데이터를 확보한다.

In [ ]:
# run_script("src/api/collect_eduinfo_data.py")

## 7. 수집 결과 확인

수집 이후 `data/processed`에 모델용 중간 산출물이 생성되었는지 확인한다.

In [ ]:
processed = ROOT / "data" / "processed"
expected = [
    "national_population_features_sgg.csv",
    "national_birth_features_sgg.csv",
    "national_migration_features_sgg.csv",
    "national_fertility_features_sgg.csv",
    "national_small_shop_sgg_summary.csv",
    "national_current_objective_closure_risk.csv",
]
for name in expected:
    path = processed / name
    print(f"{name}: {path.exists()}")